# Chapter 37 — Alignment: Turning a Predictor into an Assistant

*From Absolute Zero* — companion notebook.

Every block below is the code printed in the chapter, in the same order. Run the cells top to bottom; the output should match the book exactly. If it does not, check `requirements.txt` first, then `docs/TROUBLESHOOTING.md`.

In [1]:
!pip -q install -r https://raw.githubusercontent.com/FromAbsoluteZero/CodeBase/main/requirements.txt  # Colab only; skip locally

zsh:1: command not found: pip


## Shared setup

Imports and the objects the blocks below reuse. The chapter prints these once and then continues the same session. This cell is `code/ch37/_lib.py`.

In [2]:
import numpy as np
rng = np.random.default_rng(3)
replies = ["refuses politely", "answers briefly", "answers with detail",
           "answers with sources", "hedges vaguely", "rambles at length",
           "flatters the user", "answers, then flatters"]
true_utility = np.array([0.1, 1.4, 2.1, 2.6, -0.4, -0.9, -1.2, 1.0])
logits_ref = np.array([0.4, 0.3, 0.2, -0.6, 0.9, 1.1, 0.7, 0.1])
softmax = lambda z: np.exp(z - z.max()) / np.exp(z - z.max()).sum()
pi_ref = softmax(logits_ref)

N = 4000
a = rng.integers(0, 8, N); b = rng.integers(0, 8, N)
keep = a != b; a, b = a[keep], b[keep]
gap = true_utility[a] - true_utility[b]
a_wins = rng.random(len(a)) < 1 / (1 + np.exp(-gap))

r = np.zeros(8)
for _ in range(4000):
    p_a = 1 / (1 + np.exp(-(r[a] - r[b])))
    err = a_wins - p_a
    g = np.zeros(8); np.add.at(g, a, -err); np.add.at(g, b, err)
    r -= 0.05 * g / len(a)
r = r - r.mean()
truth = true_utility - true_utility.mean()

## The chapter code

### Block 1  (`c1.py`)

In [3]:
import numpy as np
rng = np.random.default_rng(3)

# Eight candidate replies a small model might give to one prompt.
replies = ["refuses politely", "answers briefly", "answers with detail",
           "answers with sources", "hedges vaguely", "rambles at length",
           "flatters the user", "answers, then flatters"]

# What people actually want. The model never sees this.
true_utility = np.array([0.1, 1.4, 2.1, 2.6, -0.4, -0.9, -1.2, 1.0])

# The pretrained model, before any alignment: fluent, not helpful.
logits_ref = np.array([0.4, 0.3, 0.2, -0.6, 0.9, 1.1, 0.7, 0.1])
softmax = lambda z: np.exp(z - z.max()) / np.exp(z - z.max()).sum()
pi_ref = softmax(logits_ref)

print(f"{'reply':<24}{'true utility':>13}{'pretrained p':>14}")
for r, u, p in zip(replies, true_utility, pi_ref):
    print(f"  {r:<22}{u:>+13.1f}{p:>14.3f}")
print(f"\nexpected utility of the pretrained model: "
      f"{pi_ref @ true_utility:+.3f}")

reply                    true utility  pretrained p
  refuses politely               +0.1         0.113
  answers briefly                +1.4         0.102
  answers with detail            +2.1         0.093
  answers with sources           +2.6         0.042
  hedges vaguely                 -0.4         0.186
  rambles at length              -0.9         0.228
  flatters the user              -1.2         0.153
  answers, then flatters         +1.0         0.084

expected utility of the pretrained model: +0.078


### Block 2  (`c2.py`)

In [4]:
# Show a labeller two replies; they pick one. Bradley-Terry says the
# probability of picking A over B is the logistic of the utility gap.
N = 4000
a = rng.integers(0, 8, N)
b = rng.integers(0, 8, N)
keep = a != b
a, b = a[keep], b[keep]
gap = true_utility[a] - true_utility[b]
a_wins = rng.random(len(a)) < 1 / (1 + np.exp(-gap))

print(f"{len(a):,} preference pairs collected")
print(f"labeller picked A {a_wins.mean():.1%} of the time")

# A worked example of one pair
i = 0
print(f"\n  A = {replies[a[i]]!r}  vs  B = {replies[b[i]]!r}")
print(f"  true gap {gap[i]:+.1f}  ->  P(A preferred) "
      f"{1/(1+np.exp(-gap[i])):.3f}  ->  labeller chose "
      f"{'A' if a_wins[i] else 'B'}")

3,489 preference pairs collected
labeller picked A 51.1% of the time

  A = 'flatters the user'  vs  B = 'answers with detail'
  true gap -3.3  ->  P(A preferred) 0.036  ->  labeller chose B


### Block 3  (`c3.py`)

In [5]:
# Fit a reward model: one number per reply, learned only from the choices.
r = np.zeros(8)
for step in range(4000):
    p_a = 1 / (1 + np.exp(-(r[a] - r[b])))       # model's P(A preferred)
    err = a_wins - p_a                           # gradient of log-likelihood
    g = np.zeros(8)
    np.add.at(g, a, -err)
    np.add.at(g, b,  err)
    r -= 0.05 * g / len(a)

r = r - r.mean()                                  # only gaps are identified
truth = true_utility - true_utility.mean()

print(f"{'reply':<24}{'learned r':>11}{'true':>9}")
for name, rr, tt in zip(replies, r, truth):
    print(f"  {name:<22}{rr:>+11.2f}{tt:>+9.2f}")
print(f"\ncorrelation with truth: {np.corrcoef(r, truth)[0,1]:.3f}")

reply                     learned r     true
  refuses politely            -0.48    -0.49
  answers briefly             +0.86    +0.81
  answers with detail         +1.44    +1.51
  answers with sources        +2.06    +2.01
  hedges vaguely              -1.02    -0.99
  rambles at length           -1.37    -1.49
  flatters the user           -1.77    -1.79
  answers, then flatters      +0.28    +0.41

correlation with truth: 0.998


### Block 4  (`c4.py`)

In [6]:
# RLHF: raise expected reward, but stay near the pretrained model.
# The KL penalty is what stops it collapsing onto one reply.
def rlhf(beta, steps=3000, eta=0.1):
    z = logits_ref.copy()
    for _ in range(steps):
        p = softmax(z)
        # gradient of  E[r] - beta * KL(pi || pi_ref)
        adv = r - beta * (np.log(p / pi_ref) + 1)
        z += eta * p * (adv - p @ adv)
    return softmax(z)

print(f"{'beta':>6}{'KL':>8}{'model reward':>14}"
      f"{'TRUE utility':>14}   top reply")
for beta in [10.0, 2.0, 0.5, 0.1, 0.02]:
    p = rlhf(beta)
    kl = (p * np.log(p / pi_ref)).sum()
    print(f"{beta:>6.2f}{kl:>8.2f}{p @ r:>14.2f}{p @ true_utility:>14.2f}"
          f"   {replies[p.argmax()]}")
print(f"\npretrained baseline: true utility {pi_ref @ true_utility:+.2f}")

  beta      KL  model reward  TRUE utility   top reply
 10.00    0.01         -0.36          0.22   rambles at length
  2.00    0.20          0.27          0.86   answers with detail
  0.50    1.60          1.61          2.19   answers with sources
  0.10    3.11          2.05          2.59   answers with sources
  0.02    3.13          2.05          2.59   answers with sources

pretrained baseline: true utility +0.08


### Block 5  (`c5.py`)

In [7]:
# Labellers are human. Suppose they mildly enjoy being flattered,
# so what they REWARD is not quite what serves them.
flattery = np.array([0, 0, 0, 0, 0, 0, 1, 1])
proxy_utility = true_utility + 2.2 * flattery      # what the labels reflect

gap_p = proxy_utility[a] - proxy_utility[b]
a_wins_p = rng.random(len(a)) < 1 / (1 + np.exp(-gap_p))

r_bias = np.zeros(8)
for _ in range(4000):
    p_a = 1 / (1 + np.exp(-(r_bias[a] - r_bias[b])))
    err = a_wins_p - p_a
    g = np.zeros(8); np.add.at(g, a, -err); np.add.at(g, b, err)
    r_bias -= 0.05 * g / len(a)
r_bias -= r_bias.mean()

def rlhf(reward, beta, steps=3000, eta=0.1):
    z = logits_ref.copy()
    for _ in range(steps):
        p = softmax(z)
        adv = reward - beta * (np.log(p / pi_ref) + 1)
        z += eta * p * (adv - p @ adv)
    return softmax(z)

print(f"{'beta':>6}{'reward model score':>20}"
      f"{'TRUE utility':>14}   top reply")
for beta in [2.0, 0.5, 0.1, 0.02]:
    p = rlhf(r_bias, beta)
    print(f"{beta:>6.2f}{p @ r_bias:>20.2f}{p @ true_utility:>14.2f}"
          f"   {replies[p.argmax()]}")

  beta  reward model score  TRUE utility   top reply
  2.00                0.39          0.69   answers, then flatters
  0.50                1.74          1.27   answers, then flatters
  0.10                2.01          1.00   answers, then flatters
  0.02                2.01          1.00   answers, then flatters


### Block 6  (`c6.py`)

In [8]:
# DPO: no reward model, no sampling. Train the policy straight on the pairs.
def dpo(beta, steps=6000, eta=0.5):
    z = logits_ref.copy()
    win  = np.where(a_wins, a, b)
    lose = np.where(a_wins, b, a)
    for _ in range(steps):
        p = softmax(z)
        s = np.log(p / pi_ref)             # implicit reward, up to beta
        margin = beta * (s[win] - s[lose])
        w = 1 / (1 + np.exp(margin))                # sigmoid of the negative
        g = np.zeros(8)
        np.add.at(g, win, -w); np.add.at(g, lose, w)
        g = beta * g / len(win)
        z -= eta * (g - p @ g)
    return softmax(z)

print(f"{'beta':>6}{'TRUE utility':>14}   top reply")
for beta in [1.0, 0.3, 0.1]:
    p = dpo(beta)
    print(f"{beta:>6.2f}{p @ true_utility:>14.2f}   {replies[p.argmax()]}")

p = dpo(0.1)
print(f"\n{'reply':<24}{'pretrained':>12}{'after DPO':>11}")
for name, q0, q1 in zip(replies, pi_ref, p):
    print(f"  {name:<22}{q0:>12.3f}{q1:>11.3f}")

  beta  TRUE utility   top reply


  1.00          1.57   answers with detail


  0.30          2.45   answers with sources


  0.10          2.56   answers with sources



reply                     pretrained  after DPO
  refuses politely             0.113      0.000
  answers briefly              0.102      0.003
  answers with detail          0.093      0.077
  answers with sources         0.042      0.921
  hedges vaguely               0.186      0.000
  rambles at length            0.228      0.000
  flatters the user            0.153      0.000
  answers, then flatters       0.084      0.000
